# Intermediate 04 - 등록과 위치 (Registration & Location)

이 노트북에서는:
- SIP REGISTER의 동작 원리를 배웁니다
- `save()`, `lookup()` 함수의 역할을 이해합니다
- Location 테이블의 개념을 익힙니다
- 등록 만료와 갱신 흐름을 연습합니다

---

## SIP 등록이란?

SIP UA(전화기, 앱)는 자신의 현재 위치(Contact URI)를 Registrar에 등록합니다:

```
UA: REGISTER sip:example.com
    From: <sip:1001@example.com>
    Contact: <sip:1001@192.168.1.100:5060>
    Expires: 3600

Registrar: 200 OK
    (location 테이블에 1001 → 192.168.1.100 저장)
```

이후 누군가 1001에게 전화를 걸면, Proxy는 location 테이블에서
1001의 현재 위치를 조회(`lookup`)하여 전달합니다.

## 1. REGISTER 메시지 생성

In [ ]:
%%sip REGISTER
From: <sip:1001@example.com>;tag=reg01
To: <sip:1001@example.com>
Contact: <sip:1001@192.168.1.100:5060>

In [ ]:
# REGISTER 정보 확인
$var(user) = $(fu{uri.user});
$var(contact) = $ct;
xlog("Registration request:");
xlog("  User: $var(user)");
xlog("  Contact: $var(contact)");

## 2. save() — 위치 등록

`save("location")`은 REGISTER 요청의 Contact 정보를 location 테이블에 저장합니다.

실제 Kamailio에서는 내부 DB에 저장하지만, 여기서는 시뮬레이션합니다.

```kamailio
# 실무 코드
if (!save("location")) {
    sl_reply_error();
}
```

In [ ]:
# save 시뮬레이션
$var(user) = $(fu{uri.user});
xlog("Saving registration for $var(user)");
save("location");
xlog("Registration saved successfully");
send_reply(200, "OK");

## 3. lookup() — 위치 조회

INVITE가 오면 `lookup("location")`으로 수신자의 현재 위치를 찾습니다.
성공하면 `$ru`(Request URI)가 실제 Contact URI로 바뀝니다.

```kamailio
# 실무 코드
if (!lookup("location")) {
    sl_send_reply("404", "Not Found");
    exit;
}
# $ru가 등록된 Contact URI로 변경됨
```

In [ ]:
%%sip INVITE
From: <sip:2001@example.com>;tag=call1
To: <sip:1001@example.com>

In [ ]:
# lookup 시뮬레이션
$var(callee) = $(ru{uri.user});
xlog("Looking up $var(callee) in location table");
lookup("location");

# 시뮬레이션: $ru를 등록된 Contact로 변경
$ru = "sip:1001@192.168.1.100:5060";
xlog("Found! Routing to: $ru");

## 4. 사용자 없음 — 404 처리

등록되지 않은 사용자에게 전화하면 404 응답을 보냅니다.

In [ ]:
%%reset

In [ ]:
%%sip INVITE
From: <sip:2001@example.com>;tag=call2
To: <sip:9999@example.com>

In [ ]:
# lookup 실패 시뮬레이션
$var(callee) = $(ru{uri.user});
$var(registered) = "no";

xlog("Looking up $var(callee) in location table");

if ($var(registered) == "no") {
    xlog("User $var(callee) not found in location");
    send_reply(404, "Not Found");
} else {
    xlog("Found — routing to $ru");
}

## 5. 등록 만료와 갱신

SIP 등록은 만료 시간이 있습니다. UA는 만료 전에 REGISTER를 다시 보내야 합니다.

```
등록:    REGISTER (Expires: 3600)
         ↓ 1시간 경과
만료:    location 테이블에서 제거됨
갱신:    REGISTER (Expires: 3600) — 만료 전에 재등록
```

Kamailio의 `modparam("usrloc", "default_expires", 3600)`으로 기본 만료 시간을 설정합니다.

In [ ]:
# 등록 만료 시뮬레이션
$var(expiry) = 3600;
xlog("Registration expires in $var(expiry) seconds");

# 만료 체크 시뮬레이션
$var(remaining) = 100;
if ($var(remaining) > 0) {
    xlog("Registration still valid ($var(remaining) seconds remaining)");
} else {
    xlog("Registration EXPIRED — need to re-register");
    send_reply(423, "Interval Too Brief");
}

## 6. registered() — 등록 여부 확인

`registered("location")`은 DB 조회 없이 메모리에서 등록 여부만 빠르게 확인합니다.

```kamailio
if (registered("location")) {
    # 이미 등록됨 — 기존 등록 업데이트
} else {
    # 새 등록
}
```

In [ ]:
# registered 체크 시뮬레이션
$var(user) = "1001";
$var(is_registered) = "yes";

xlog("Checking if $var(user) is registered");

if ($var(is_registered) == "yes") {
    xlog("$var(user) is already registered — updating contact");
} else {
    xlog("$var(user) is a new registration");
}

## 7. 전체 등록→조회→통화 흐름

실무 SIP 프록시의 핵심 흐름을 종합합니다:

In [ ]:
# Step 1: 1001이 등록
%%sip REGISTER
From: <sip:1001@example.com>;tag=r1
To: <sip:1001@example.com>
Contact: <sip:1001@10.0.0.5:5060>

In [ ]:
$var(user) = $(fu{uri.user});
xlog("[REGISTER] Saving location for $var(user)");
save("location");
send_reply(200, "OK");

In [ ]:
# Step 2: 2001이 1001에게 통화 요청
%%sip INVITE
From: <sip:2001@example.com>;tag=call01
To: <sip:1001@example.com>

In [ ]:
$var(callee) = $(ru{uri.user});
xlog("[INVITE] Looking up $var(callee)");
lookup("location");

# 시뮬레이션: location 조회 결과로 $ru 업데이트
$ru = "sip:1001@10.0.0.5:5060";
xlog("[INVITE] Routing to: $ru");
record_route();
t_relay();

---

### 요약

| 개념 | 설명 |
|------|------|
| REGISTER | UA가 자신의 위치를 서버에 등록 |
| save("location") | Contact 정보를 location 테이블에 저장 |
| lookup("location") | 수신자 위치 조회, 성공 시 $ru 변경 |
| registered("location") | 등록 여부 빠른 확인 |
| Expires | 등록 만료 시간 (초) |
| 404 Not Found | 등록되지 않은 사용자 |
| 423 Interval Too Brief | 만료 시간이 너무 짧음 |

다음 노트북: **Advanced/01-dialog-and-failover.ipynb** →